# 04b — Social Media Charts (Altair)

Publication-ready charts using the @unwelcomedata brand palette (Coolors).
These render inline for preview and export to `outputs/` as exact-dimension PNGs.

Charts:
1. Scatter — consumption vs fatality rate (by region)
2. Ranked bars — top/bottom 10 states (per VMT)
3. Comparison — IID vs non-IID
4. Trend — national fatalities 2015–2020
5. Choropleth — fatality rate per 100M VMT
6. Choropleth — felony status (category)

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import yaml

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.viz_social import (
    social_scatter,
    social_ranked_bars,
    social_comparison,
    social_trend,
    social_choropleth,
    social_bubble_choropleth,
    social_bivariate_choropleth,
    save_social,
)

with open(PROJECT / "config.yaml") as f:
    cfg = yaml.safe_load(f)

df = pd.read_parquet(PROJECT / "export" / "dui_by_state_v1.parquet")
print(f"Project: {PROJECT.name}")
print(f"Master table: {df.shape[0]} states x {df.shape[1]} columns")

## 1. Scatter — Consumption vs Fatality Rate

In [ ]:
chart = social_scatter(
    df,
    x="ethanol_per_capita_gallons_2022",
    y="alcohol_fatality_rate_per_100m_vmt",
    color_by="region",
    title="Alcohol Consumption vs Impaired-Driving Deaths",
    subtitle="Per-VMT fatality rate controls for driving exposure. Each dot is a state.",
    source="NIAAA 2022, NHTSA FARS 2024, FHWA VMT 2022",
    xlabel="Per capita ethanol (gallons, 2022)",
    ylabel="Alcohol fatalities per 100M VMT",
    label_col="state_abbr",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_scatter_consumption_vs_fatality", preset="twitter_landscape")
chart

## 2. Ranked Bars — Top/Bottom 10 (per VMT)

In [ ]:
chart = social_ranked_bars(
    df,
    x="state_name",
    y="alcohol_fatality_rate_per_100m_vmt",
    title="Worst & Best States for Impaired-Driving Deaths",
    subtitle="Alcohol fatalities per 100M vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    top_n=10,
    bottom_n=10,
    preset="instagram_portrait",
)
save_social(chart, cfg, "social_ranked_top_bottom_10", preset="instagram_portrait")
chart

## 3. IID vs Non-IID Comparison

In [ ]:
df["iid_group"] = df["all_offender_iid"].map({1: "IID for all offenders", 0: "No universal IID"})

chart = social_comparison(
    df,
    group_col="iid_group",
    value_col="alcohol_fatality_rate_per_100m_vmt",
    title="Does Mandatory IID Reduce Impaired-Driving Deaths?",
    subtitle="Mean alcohol fatality rate per 100M VMT by IID policy",
    source="NHTSA FARS 2024, FHWA VMT 2022, IIHS/GHSA",
    colors={"IID for all offenders": "#2A9D8F", "No universal IID": "#E76F51"},
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_comparison_iid", preset="twitter_landscape")
chart

## 4. Trend — National Fatalities 2015–2020

In [ ]:
trends = pd.read_parquet(PROJECT / "export" / "dui_trends_2015_2020.parquet")
national = trends.groupby("year", as_index=False).agg(
    impaired_fatalities=("impaired_fatalities_any", "sum"),
    total_fatalities=("total_fatalities", "sum"),
)

chart = social_trend(
    national,
    x="year",
    y="impaired_fatalities",
    title="National Alcohol-Impaired Fatalities (2015–2020)",
    subtitle="Methodology-consistent FARS coding window",
    source="NHTSA FARS 2015–2020",
    xlabel="Year",
    ylabel="Impaired-driving fatalities",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_trend_national_2015_2020", preset="twitter_landscape")
chart

## 5. Choropleth — Fatality Rate per 100M VMT

In [ ]:
chart = social_choropleth(
    df,
    column="alcohol_fatality_rate_per_100m_vmt",
    title="Alcohol-Impaired Fatality Rate by State",
    subtitle="Deaths per 100 million vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    mode="heat",
    legend_title="Deaths per 100M VMT",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_map_fatality_rate_vmt", preset="twitter_landscape")
chart

## 6. Choropleth — Felony Status (category)

In [ ]:
df["felony_label"] = df["first_offense_felony"].map({1.0: "Can be felony", 0.0: "Always misdemeanor"})
df.loc[df["felony_label"].isna(), "felony_label"] = "Unknown"

chart = social_choropleth(
    df,
    column="felony_label",
    title="First-Offense DUI: Felony Possible?",
    subtitle="States where first DUI can be charged as felony",
    source="NCSL DUI/DWI criminal status laws",
    mode="category",
    category_colors={"Can be felony": "#E76F51", "Always misdemeanor": "#2A9D8F", "Unknown": "#E5E7EB"},
    legend_title="First-offense status",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_map_felony_status", preset="twitter_landscape")
chart

## Quick exploration

One-liner functions to explore any columns interactively. Just call and view.

In [ ]:
def quick_map(column, title=None, mode="heat", **kwargs):
    """Choropleth any column. mode='heat' for numeric, 'category' for discrete."""
    title = title or column.replace('_', ' ').title()
    return social_choropleth(df, column=column, title=title, mode=mode, preset='twitter_landscape', **kwargs)


def quick_scatter(x, y, title=None, color_by='region', **kwargs):
    """Scatter any two numeric columns. Colored by region by default."""
    title = title or f"{x.replace('_',' ').title()} vs {y.replace('_',' ').title()}"
    return social_scatter(df, x=x, y=y, color_by=color_by, title=title, preset='twitter_landscape', **kwargs)


def quick_bars(y, x='state_name', title=None, top_n=10, bottom_n=0, **kwargs):
    """Ranked bar chart. Shows top_n (and optionally bottom_n) states."""
    title = title or f"Top {top_n} States by {y.replace('_',' ').title()}"
    return social_ranked_bars(df, x=x, y=y, title=title, top_n=top_n, bottom_n=bottom_n, preset='instagram_portrait', **kwargs)


def quick_compare(group_col, value_col, title=None, **kwargs):
    """Compare group means for any grouping column vs any numeric column."""
    title = title or f"{value_col.replace('_',' ').title()} by {group_col.replace('_',' ').title()}"
    return social_comparison(df, group_col=group_col, value_col=value_col, title=title, preset='twitter_landscape', **kwargs)


def quick_trend(x='year', y='impaired_fatalities', data=None, title=None, **kwargs):
    """Line trend chart. Pass a different DataFrame via data= if needed."""
    title = title or f"{y.replace('_',' ').title()} over {x.replace('_',' ').title()}"
    src = data if data is not None else df
    return social_trend(src, x=x, y=y, title=title, preset='twitter_landscape', **kwargs)


def quick_bubble_map(color_col, size_col, title=None, **kwargs):
    """Choropleth with fill color + bubble overlay showing two variables."""
    title = title or f"{color_col.replace('_',' ').title()} (color) + {size_col.replace('_',' ').title()} (size)"
    return social_bubble_choropleth(df, color_col=color_col, size_col=size_col, title=title, preset='twitter_landscape', **kwargs)


def quick_bivariate_map(x_col, y_col, title=None, **kwargs):
    """Bivariate choropleth — two variables encoded as a 3x3 color grid."""
    title = title or f"{x_col.replace('_',' ').title()} vs {y_col.replace('_',' ').title()}"
    return social_bivariate_choropleth(df, x_col=x_col, y_col=y_col, title=title, preset='twitter_landscape', **kwargs)


print('Quick functions ready: quick_map, quick_scatter, quick_bars, quick_compare, quick_trend, quick_bubble_map, quick_bivariate_map')

In [ ]:
def cols():
    """Print all columns in the master table, grouped by type."""
    numeric = [c for c in df.columns if df[c].dtype in ('float64', 'int64')]
    categorical = [c for c in df.columns if df[c].dtype == 'object']
    print(f'=== NUMERIC ({len(numeric)}) ===')
    for c in numeric:
        print(f'  {c}')
    print(f'\n=== CATEGORICAL ({len(categorical)}) ===')
    for c in categorical:
        vals = df[c].nunique()
        print(f'  {c}  ({vals} unique)')

cols()

In [ ]:
# --- Try it: consumption vs arrest rate ---
quick_scatter('ethanol_per_capita_gallons_2022', 'dui_arrest_rate_per_100k',
              title='Alcohol Consumption vs DUI Arrest Rate')

In [ ]:
# More examples (uncomment any):
# quick_map('dui_arrest_rate_per_100k', title='DUI Arrest Rate per 100k')
# quick_map('ethanol_per_capita_gallons_2022', title='Per Capita Alcohol Consumption')
# quick_scatter('dui_arrest_rate_per_100k', 'alcohol_fatality_rate_per_100m_vmt')
# quick_bars('dui_arrest_rate_per_100k', top_n=10, bottom_n=10)
# quick_compare('iid_group', 'dui_arrest_rate_per_100k')
# quick_map('pct_suspended', title='% Drivers on Suspended License in Fatal Crashes')

## Two-variable maps

Compare two variables on one map using either bubbles or bivariate coloring.

In [ ]:
# Option A: Bubble map — color = consumption, bubble size = % traffic deaths from alcohol
quick_bubble_map(
    'ethanol_per_capita_gallons_2022',
    'pct_traffic_deaths_alcohol',
    title='Alcohol Consumption (color) vs % Traffic Deaths from Alcohol (bubble)',
    color_legend='Per capita ethanol (gal)',
    size_legend='% traffic deaths alcohol',
)

In [ ]:
# Option B: Bivariate choropleth — 3x3 color grid
# Bottom-left = low consumption + low alcohol death share
# Top-right = high consumption + high alcohol death share
quick_bivariate_map(
    'ethanol_per_capita_gallons_2022',
    'pct_traffic_deaths_alcohol',
    title='Bivariate: Consumption vs % Traffic Deaths from Alcohol',
    x_label='Consumption →',
    y_label='% Deaths Alcohol →',
)